# RocketCEA and Cantera for Propellant and Gas Dynamics

## Install packages

In [1]:
!pip install rocketcea cantera

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 13.6 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 78.1 MB/s eta 0:00:00:00:0100:01
  Created wheel for rocketcea: filename=rocketcea-1.2.3-cp312-cp312-linux_x86_64.whl size=5127978 sha256=586dc0650480c8b7c93f18f77e2158698e1878f6a80a5313da37dc95d94be78e
  Stored in directory: /root/.cache/pip/wheels/57/0a/e7/3785f7e8c0bc86eb2563ccd5f8e37e59194d2cbba140deddb1
Successfully built rocketcea


## Add equilibrium chemistry to RocketCEA

In [2]:
from pathlib import Path
import rocketcea.cea_obj

path = Path(rocketcea.cea_obj.__file__)

text = path.read_text()

old = 'set_py_cea_line(N," rocket %s  %s"%(eqfrStr,pc_str) + subar_str + pcope_str + eps_str )'
new = 'set_py_cea_line(N," rocket %s ions  %s"%(eqfrStr,pc_str) + subar_str + pcope_str + eps_str )'

if old not in text:
    print("Target line NOT found — nothing changed.")
else:
    # Make a backup first
    backup = path.with_suffix(".py.backup")
    backup.write_text(text)

    # Patch it
    path.write_text(text.replace(old, new, 1))

    print("Patched:", path)
    print("Backup:", backup)

USER_HOME_DIR=/root
Patched: /usr/local/lib/python3.12/dist-packages/rocketcea/cea_obj.py
Backup: /usr/local/lib/python3.12/dist-packages/rocketcea/cea_obj.py.backup


## Find Heat of Formation from Quantum ESPRESSO output and ```.xyz``` file

### Refer to my notebook of Quantum ESPRESSO here: https://www.kaggle.com/code/pcsciprav/qe-guide-main-branch-quantum-espresso-on-gpu

In [ ]:
import math, re, urllib.request
from collections import Counter

XYZ_PATH = "/path/to/mol.xyz"
QE_OUT_PATH = "/path/to/scf.out"
E_ATOMS_RY = {"C": 0.0, "H": 0.0, "N": 0.0, "O": 0.0}

RY_TO_KJMOL = 1312.7496997
JANAF = "https://janaf.nist.gov/tables/{el}-{n:03d}.txt"

RADII = {"H": 0.31, "B": 0.84, "C": 0.76, "N": 0.71, "O": 0.66, "F": 0.57,
         "Al": 1.21, "Si": 1.11, "P": 1.07, "S": 1.05, "Cl": 1.02}
ZPE_XH = 29.0
ZPE_HEAVY = {1: 9.0, 2: 14.0, 3: 16.0}


def fetch_janaf_dhf0(el, max_n=80):
    for n in range(1, max_n + 1):
        try:
            req = urllib.request.Request(JANAF.format(el=el, n=n), headers={"User-Agent": "Mozilla/5.0"})
            txt = urllib.request.urlopen(req, timeout=20).read().decode("latin-1")
        except Exception:
            continue
        lines = [l for l in txt.splitlines() if l.strip() and not l.startswith("---")]
        if not lines or lines[0].split("\t")[-1].strip() != f"{el}1(g)":
            continue
        for l in lines:
            f = l.split()
            if f[0] == "0":
                return float(f[5])
    raise SystemExit(f"Atomic data not found on JANAF for {el}")


def read_xyz(path):
    with open(path) as f:
        lines = f.read().splitlines()
    n = int(lines[0].split()[0])
    atoms = []
    for l in lines[2:2 + n]:
        p = l.split()
        atoms.append((p[0].capitalize(), float(p[1]), float(p[2]), float(p[3])))
    return atoms


def read_qe_energy(path):
    with open(path) as f:
        m = re.findall(r"^!\s+total energy\s+=\s+(-?\d+\.\d+)\s+Ry", f.read(), re.M)
    if not m:
        raise SystemExit("No final total energy found in QE output")
    return float(m[-1])


def estimate_zpe(atoms):
    zpe = 0.0
    for i in range(len(atoms)):
        for j in range(i + 1, len(atoms)):
            ei, xi, yi, zi = atoms[i]
            ej, xj, yj, zj = atoms[j]
            d = math.dist((xi, yi, zi), (xj, yj, zj))
            ratio = d / (RADII.get(ei, 1.0) + RADII.get(ej, 1.0))
            if ratio > 1.25:
                continue
            if "H" in (ei, ej):
                zpe += ZPE_XH
            elif ratio < 0.82:
                zpe += ZPE_HEAVY[3]
            elif ratio < 0.95:
                zpe += ZPE_HEAVY[2]
            else:
                zpe += ZPE_HEAVY[1]
    return zpe


def main():
    atoms = read_xyz(XYZ_PATH)
    comp = Counter(a[0] for a in atoms)
    e_mol = read_qe_energy(QE_OUT_PATH)

    missing = [el for el in comp if el not in E_ATOMS_RY]
    if missing:
        raise SystemExit(f"Add isolated atom energy (Ry) to E_ATOMS_RY for: {' '.join(missing)}")
    dhf_atom = {el: fetch_janaf_dhf0(el) for el in comp}

    zpe = estimate_zpe(atoms)
    e_atomization = (sum(n * E_ATOMS_RY[el] for el, n in comp.items()) - e_mol) * RY_TO_KJMOL
    dhf = sum(n * dhf_atom[el] for el, n in comp.items()) - (e_atomization - zpe)

    print("Estimated ZPE kJ per mol", round(zpe, 1))
    print("Heat of formation kJ per mol", round(dhf, 3))


if __name__ == "__main__":
    main()

## Run RocketCEA

In [9]:
from rocketcea.cea_obj import CEA_Obj
from rocketcea import cea_obj

card_str = """
fuel C2N19O2 C 2 N 19 O 2  wt=100.0
h,kj/mol=789  t(k)=298.15
"""

cea_obj.add_new_fuel("MyCustomPropellant", card_str)

ispObj = CEA_Obj(propName="MyCustomPropellant")

Pc = 1000.0
eps = 100.0

final_text = ispObj.get_full_cea_output(
    Pc=Pc,
    eps=eps
)

_, _, _, mw_exhaust, gamma = ispObj.get_IvacCstrTc_ChmMwGam(
    Pc=Pc,
    eps=eps
)

with open("propellant_products.txt", "w") as f:
    f.write(final_text)

print(f"Exhaust MW: {mw_exhaust}, Gamma: {gamma}")

Exhaust MW: 28.013380528620182, Gamma: 1.2899978880032077


### Check the output

In [10]:
!cat /kaggle/working/propellant_products.txt


 *******************************************************************************

         NASA-GLENN CHEMICAL EQUILIBRIUM PROGRAM CEA, OCTOBER 18, 2002
                   BY  BONNIE MCBRIDE AND SANFORD GORDON
      REFS: NASA RP-1311, PART I, 1994 AND NASA RP-1311, PART II, 1996

 *******************************************************************************



 reac
  name C2N19O2 C 2 N 19 O 2  wt=100.0
  h,kj/mol=789  t(k)=298.15
  
 prob case=RocketCEA,
  rocket equilibrium   p,psia=1000.000000,  supar=100.000000,
  
  
  
 output calories  transport
 end

 OPTIONS: TP=F  HP=F  SP=F  TV=F  UV=F  SV=F  DETN=F  SHOCK=F  REFL=F  INCD=F
 RKT=T  FROZ=F  EQL=T  IONS=F  SIUNIT=F  DEBUGF=F  SHKDBG=F  DETDBG=F  TRNSPT=T

 TRACE= 0.00E+00  S/R= 0.000000E+00  H/R= 0.000000E+00  U/R= 0.000000E+00

 Pc,BAR =    68.947304

 Pc/P =

 SUBSONIC AREA RATIOS =

 SUPERSONIC AREA RATIOS =   100.0000

 NFZ=  1  Mdot/Ac= 0.000000E+00  Ac/At= 0.000000E+00

    REACTANT          WT.FRAC   (ENERGY/R),K   

## Set up Cantera

In [ ]:
%%bash

cat > propellant.yaml << 'EOF'
description: |
  Equilibrium model for C6N32O6F21H9 monopropellant.
  Custom reactant enthalpy from bond-additivity estimate.
  Product species imported from Cantera's nasa_gas.yaml database.

phases:
- name: gas
  thermo: ideal-gas

  elements: [C, N, O, F, H]

  species:
  - species: [MyCustomPropellant]

  - nasa_gas.yaml/species:
    - H
    - H2
    - O
    - O2
    - OH
    - H2O
    - N
    - N2
    - NO
    - F
    - F2
    - HF
    - C
    - CO
    - CO2
    - CN
    - HCN
    - CF
    - CF2
    - CF3
    - CF4
    - COF2
    - COF
    - FCN
    - NF

  kinetics: none

  state:
    T: 298.15 K
    P: 1 atm

species:

- name: MyCustomPropellant

  composition:
    C: 6
    N: 32
    O: 6
    F: 21
    H: 9

  thermo:
    model: constant-cp
    T0: 298.15 K

    # 2650 matches the CEA run. Use 2250 or 3050 for the low and high cases
    h0: 3050 kJ/mol

    # Placeholder, same 74 atoms as before
    s0: 1224.5588 J/mol/K  
EOF

In [ ]:
import cantera as ct

species = ct.Species.list_from_file("nasa_gas.yaml")
names = {sp.name for sp in species}

wanted = [
    "H", "H2", "O", "O2", "OH", "H2O",
    "N", "N2", "NO",
    "F", "F2", "HF",
    "C", "CO", "CO2", "CN", "HCN"
]

for name in wanted:
    print(f"{name:5s}", "OK" if name in names else "MISSING")

## Get properties from Cantera

In [ ]:
import sys
import contextlib
import cantera as ct


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, s):
        for st in self.streams:
            st.write(s)
        return len(s)

    def flush(self):
        for st in self.streams:
            st.flush()


OUT = "/kaggle/working/cantera_output.txt"

with open(OUT, "w") as f, contextlib.redirect_stdout(Tee(sys.stdout, f)):
    gas = ct.Solution("/kaggle/working/propellant.yaml")

    T0 = 298.15
    Pc = 1000.0 * 6894.757293168

    gas.TPX = T0, Pc, {"MyCustomPropellant": 1.0}
    print("Initial state")
    print(gas.report())

    gas.equilibrate(
        "HP",
        solver="vcs",
        max_steps=100000000,
        max_iter=1000000,
        estimate_equil=-1,
        log_level=1
    )

    print("Equilibrium state")
    print(gas.report(threshold=0.0))
    print("Chamber temperature K", gas.T)
    print("Chamber pressure Pa", gas.P)
    print("Mean molecular weight", gas.mean_molecular_weight)
    print("Enthalpy J per kg", gas.enthalpy_mass)
    print("Entropy J per kg per K", gas.entropy_mass)
    print("Mole fractions")
    for name, x in sorted(zip(gas.species_names, gas.X), key=lambda p: -p[1]):
        print(name, x)

print("Saved to", OUT)